<a href="https://colab.research.google.com/github/Markkwell/Project-YpYt/blob/main/project1_by_Marko_Deko.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Your path Youe trials





In [ ]:
"""
Your Path Your Trials
Система анализа Steam-отзывов
"""

import re

import numpy as np
import pandas as pd


DATASET_PATH = "steam_reviews.csv"

REQUIRED_COLUMNS = (
    "app_id",
    "app_name",
    "review_text",
    "review_score",
    "review_votes"
)


def load_dataset(path: str) -> pd.DataFrame:
    """
    Загружает датасет и проверяет структуру.
    """

    try:
        df = pd.read_csv(path)

    except FileNotFoundError:
        raise ValueError(
            f"Файл не найден: {path}"
        )

    if df.empty:
        raise ValueError(
            "Датасет пуст."
        )

    missing_columns = [
        column
        for column in REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Отсутствуют столбцы: "
            f"{missing_columns}"
        )

    return df


def clean_text(text: str) -> str:
    """
    Очищает текст от символов.
    """

    text = str(text).lower()

    text = re.sub(
        r"[^a-zA-Z ]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def prepare_reviews(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Подготавливает отзывы.
    """

    df = df.copy()

    df["app_name"] = (
        df["app_name"]
        .fillna("Unknown")
    )

    df["review_text"] = (
        df["review_text"]
        .fillna("")
    )

    df["clean_review"] = [
        clean_text(text)
        for text in df["review_text"]
    ]

    df["sentiment"] = np.where(
        df["review_score"] > 0,
        "Positive",
        "Negative"
    )

    return df


def get_game_rating(
    df: pd.DataFrame
) -> pd.DataFrame:
    """
    Считает статистику отзывов
    по каждой игре.
    """

    grouped = (
        df.groupby(
            ["app_name", "sentiment"]
        )
        .size()
        .unstack(fill_value=0)
    )

    grouped["Positive"] = (
        grouped.get(
            "Positive",
            0
        )
    )

    grouped["Negative"] = (
        grouped.get(
            "Negative",
            0
        )
    )

    grouped["ratio"] = (
        grouped["Positive"] /
        (
            grouped["Negative"] + 1
        )
    )

    grouped["total_reviews"] = (
        grouped["Positive"] +
        grouped["Negative"]
    )

    return grouped.reset_index()


def show_games(
    df: pd.DataFrame
) -> None:
    """
    Показывает игры.
    """

    games = (
        df["app_name"]
        .dropna()
        .unique()
    )

    print("\nСписок игр:\n")

    for game in games[:20]:
        print(game)


def show_reviews(
    df: pd.DataFrame,
    sentiment: str
) -> None:
    """
    Показывает отзывы
    по типу.
    """

    reviews = (
        df[
            df["sentiment"] ==
            sentiment
        ][
            [
                "app_name",
                "review_text"
            ]
        ]
        .head(5)
    )

    print(
        f"\n{sentiment} reviews:\n"
    )

    for _, row in reviews.iterrows():

        print(
            f"Игра: "
            f"{row['app_name']}"
        )

        print(
            f"Отзыв: "
            f"{row['review_text']}"
        )

        print("-" * 40)


def search_game(
    df: pd.DataFrame
) -> None:
    """
    Поиск игры.
    """

    game_name = input(
        "Введите название игры: "
    ).strip()

    result = df[
        df["app_name"]
        .str.contains(
            game_name,
            case=False,
            na=False
        )
    ]

    if result.empty:
        print(
            "Игра не найдена."
        )
        return

    print(
        result[
            [
                "app_name",
                "sentiment"
            ]
        ]
        .head(10)
    )


def top_r_games(
    df: pd.DataFrame
) -> None:
    """
    5 игр с лучшим
    Positive/Negative.
    """

    stats = get_game_rating(df)

    result = (
        stats.sort_values(
            by="ratio",
            ascending=False
        )
        .head(5)
    )

    print("\nTop R:\n")

    for _, row in result.iterrows():

        print(
            f"{row['app_name']} | "
            f"Positive="
            f"{row['Positive']} | "
            f"Negative="
            f"{row['Negative']} | "
            f"P/N="
            f"{row['ratio']:.2f}"
        )


def lead_comm(
    df: pd.DataFrame
) -> None:
    """
    5 игр с самым большим
    числом отзывов.
    """

    stats = get_game_rating(df)

    result = (
        stats.sort_values(
            by="total_reviews",
            ascending=False
        )
        .head(5)
    )

    print(
        "\nLead comm:\n"
    )

    for _, row in result.iterrows():

        print(
            f"{row['app_name']} | "
            f"Отзывы: "
            f"{row['total_reviews']}"
        )


def random_comm(
    df: pd.DataFrame,
    game_name: str,
    review_count: int
) -> None:
    """
    Показывает случайные
    отзывы игры.
    """

    result = df[
        df["app_name"]
        .str.contains(
            game_name,
            case=False,
            na=False
        )
    ]

    if result.empty:
        print(
            "Игра не найдена."
        )
        return

    sample_size = min(
        review_count,
        len(result)
    )

    sample = result.sample(
        sample_size,
        random_state=np.random.randint(
            0,
            1000
        )
    )

    print(
        "\nRandom comments:\n"
    )

    for _, row in sample.iterrows():

        print(
            f"[{row['sentiment']}] "
            f"{row['review_text']}"
        )

        print(
            "-" * 40
        )


def game_list(
    df: pd.DataFrame
) -> None:
    """
    Список игр
    без повторов.
    """

    games = sorted(
        df["app_name"]
        .dropna()
        .unique()
    )

    print(
        "\nGame list:\n"
    )

    for game in games:
        print(game)


def compare_games(
    df: pd.DataFrame,
    game_1: str,
    game_2: str
) -> None:
    """
    Сравнивает игры.
    """

    stats = get_game_rating(df)

    games = stats[
        stats["app_name"]
        .isin(
            [
                game_1,
                game_2
            ]
        )
    ]

    if len(games) < 2:

        print(
            "Одна из игр "
            "не найдена."
        )

        return

    print(
        "\nCompare:\n"
    )

    for _, row in games.iterrows():

        print(
            f"{row['app_name']} | "
            f"Positive="
            f"{row['Positive']} | "
            f"Negative="
            f"{row['Negative']} | "
            f"P/N="
            f"{row['ratio']:.2f}"
        )


def print_menu() -> None:
    """
    Показывает меню.
    """

    print("\nКоманды:")

    print(
        "1 - показать игры"
    )

    print(
        "2 - positive reviews"
    )

    print(
        "3 - negative reviews"
    )

    print(
        "Top R - показать 5 игр имеющих наилучшее соотношение положительных/отрицательных отзывов"
    )

    print(
        "Lead comm - показать 5 игр, имеющих наибольшее колличество отзывов"
    )

    print(
        "Random comm (game) (число отзывов) - показать n колличество отзывов по игре, выбранной игроком "
    )

    print(
        "Game list - показать лист с играми (название каждой игры только 1 раз)"
    )

    print(
        "Compare (game) (game) - сравнить соотношение положительных/отрицательных отзывов выбранных игр"
    )

    print(
        "0 - выход"
    )


def handle_command(
    command: str,
    df: pd.DataFrame
) -> bool:
    """
    Обрабатывает команды.
    """

    parts = command.split()

    try:

        if command == "1":
            show_games(df)

        elif command == "2":
            show_reviews(
                df,
                "Positive"
            )

        elif command == "3":
            show_reviews(
                df,
                "Negative"
            )

        elif command == "4":
            search_game(df)

        elif (
            command.lower() ==
            "top r"
        ):
            top_r_games(df)

        elif (
            command.lower() ==
            "lead comm"
        ):
            lead_comm(df)

        elif (
            command.lower() ==
            "game list"
        ):
            game_list(df)

        elif (
            len(parts) >= 4
            and
            parts[0].lower()
            == "random"
            and
            parts[1].lower()
            == "comm"
        ):

            game = " ".join(
                parts[2:-1]
            )

            count = int(
                parts[-1]
            )

            random_comm(
                df,
                game,
                count
            )

        elif (
            len(parts) >= 3
            and
            parts[0].lower()
            == "compare"
        ):

            compare_games(
                df,
                parts[1],
                parts[2]
            )

        elif command == "0":

            print(
                "Работа завершена."
            )

            return False

        else:

            print(
                "Неизвестная "
                "команда."
            )

    except ValueError:

        print(
            "Ошибка данных."
        )

    except KeyError:

        print(
            "Ошибка столбцов."
        )

    except Exception as error:

        print(
            f"Ошибка: "
            f"{error}"
        )

    return True


def main() -> None:
    """
    Главная функция.
    """

    try:

        df = load_dataset(
            DATASET_PATH
        )

        prepared_df = (
            prepare_reviews(df)
        )

        print(
            f"\nЗагружено "
            f"отзывов: "
            f"{len(prepared_df)}"
        )

        while True:

            print_menu()

            command = input(
                "\nВведите "
                "команду: "
            )

            if not handle_command(
                command,
                prepared_df
            ):
                break

    except ValueError as error:

        print(
            f"Ошибка: "
            f"{error}"
        )

    except Exception as error:

        print(
            f"Критическая "
            f"ошибка: "
            f"{error}"
        )


if __name__ == "__main__":
    main()